# Fase 1: Resultados de clasificacion full

Este notebook consolida exclusivamente los artefactos `full` generados por los notebooks CXR y CT. Las ejecuciones reducidas no se cargan ni se usan para rankings, graficas o conclusiones de la memoria.


## 1. Setup

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib_cache'))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, auc, roc_curve
from sklearn.preprocessing import label_binarize

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

## 2. Carga de resúmenes

In [ ]:
results_root = PROJECT_ROOT / 'results' / 'classification'
summary_paths = sorted(results_root.glob('*/*_full_summary.json'))

summaries = []
for summary_path in summary_paths:
    summary = json.loads(summary_path.read_text())
    if summary.get('run_mode') != 'full':
        continue
    summary['summary_path'] = str(summary_path)
    summaries.append(summary)

summary_df = pd.DataFrame(summaries)
if summary_df.empty:
    raise FileNotFoundError(f'No se encontraron resúmenes full en {results_root}. Ejecuta primero los notebooks 02/03 o scripts/run_phase1_full.py')

summary_df = summary_df.sort_values(['dataset', 'architecture', 'balance_strategy']).reset_index(drop=True)
display(summary_df)


## 3. Tabla comparativa principal

In [ ]:
metric_cols = ['accuracy', 'f1_macro', 'f1_weighted', 'auc_roc_macro']
comparison = summary_df[['dataset', 'run_mode', 'architecture', 'balance_strategy', *metric_cols]].copy()
display(comparison)

best_by_dataset = comparison.sort_values(['dataset', 'f1_macro'], ascending=[True, False]).groupby('dataset').head(1)
display(best_by_dataset)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=comparison,
    x='architecture',
    y='f1_macro',
    hue='dataset',
    ax=ax,
)
ax.set_ylim(0, 1)
ax.set_title('F1-macro por arquitectura y modalidad')
ax.set_xlabel('Arquitectura')
ax.set_ylabel('F1-macro')
plt.tight_layout()

## 4. Matrices de confusión disponibles

In [ ]:
def artifact_prefix(summary_path: str) -> Path:
    path = Path(summary_path)
    return path.with_name(path.name.replace('_summary.json', ''))

for _, row in summary_df.iterrows():
    confusion_path = artifact_prefix(row['summary_path']).with_name(artifact_prefix(row['summary_path']).name + '_confusion_matrix.csv')
    if not confusion_path.exists():
        continue
    confusion = pd.read_csv(confusion_path)
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    sns.heatmap(confusion, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    ax.set_title(f"{row['dataset']} | {row['architecture']} | {row['balance_strategy']} | {row['run_mode']}")
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Real')
    plt.tight_layout()

## 5. Curvas ROC multiclase si hay predicciones

In [ ]:
for _, row in summary_df.iterrows():
    prefix = artifact_prefix(row['summary_path'])
    predictions_path = prefix.with_name(prefix.name + '_predictions.csv')
    if not predictions_path.exists():
        continue

    pred_df = pd.read_csv(predictions_path)
    prob_cols = [col for col in pred_df.columns if col.startswith('prob_')]
    if len(prob_cols) < 2:
        continue

    y_true_bin = label_binarize(pred_df['y_true'], classes=list(range(len(prob_cols))))
    y_score = pred_df[prob_cols].to_numpy()

    fig, ax = plt.subplots(figsize=(5.8, 4.6))
    for class_idx, prob_col in enumerate(prob_cols):
        if y_true_bin[:, class_idx].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_true_bin[:, class_idx], y_score[:, class_idx])
        ax.plot(fpr, tpr, label=f'class {class_idx} AUC={auc(fpr, tpr):.2f}')
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray')
    ax.set_title(f"ROC OvR: {row['dataset']} | {row['architecture']} | {row['run_mode']}")
    ax.set_xlabel('False positive rate')
    ax.set_ylabel('True positive rate')
    ax.legend(loc='lower right')
    plt.tight_layout()

## 6. Lectura cross-modal

In [ ]:
cross_modal = comparison.pivot_table(
    index=['architecture', 'balance_strategy', 'run_mode'],
    columns='dataset',
    values=['accuracy', 'f1_macro', 'auc_roc_macro'],
)
display(cross_modal)

analysis_notes = pd.DataFrame([
    {
        'lectura': 'Comparar CXR vs CT solo como rendimiento metodológico',
        'motivo': 'Las etiquetas CXR son diagnósticas y las CT representan severidad agrupada; no son clases clínicas equivalentes.',
    },
    {
        'lectura': 'Priorizar F1-macro en presencia de desbalanceo',
        'motivo': 'Accuracy puede ocultar errores en clases minoritarias como CT-3+.',
    },
    {
        'lectura': 'Usar solamente ejecuciones full en resultados finales',
        'motivo': 'Las tablas, graficas y conclusiones de la memoria deben proceder de entrenamientos completos.',
    },
])
display(analysis_notes)
